In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

county = pd.read_csv(
    "../data/processed/coverage_ratio_2024.csv",
    dtype={"fips": str}
)

county["fips"] = (
    county["fips"]
    .str.zfill(5)
)

print("Counties:", len(county))
display(
    county[
        [
            "County",
            "fips",
            "visits",
            "expected_visits",
            "coverage_ratio"
        ]
    ].head()
)

Counties: 86


,County,fips,visits,expected_visits,coverage_ratio
0,Aitkin,27001,330.916667,739.739053,0.447342
1,Anoka,27003,19465.083333,14332.082579,1.358148
2,Becker,27005,543.750000,1560.666580,0.348409
3,Beltrami,27007,1250.333333,2141.679498,0.583810
4,Benton,27009,673.083333,1794.669680,0.375046


In [2]:
county["annual_visits"] = (
    county["visits"] * 12
).round().astype(int)

county["expected_annual_visits"] = (
    county["expected_visits"] * 12
)

county["log_expected_annual"] = np.log(
    county["expected_annual_visits"]
)

m0 = smf.negativebinomial(
    formula="annual_visits ~ 1",
    data=county,
    offset=county["log_expected_annual"]
).fit(disp=False)

typical_center = np.exp(
    m0.params["Intercept"]
)

print(
    "Typical-county coverage ratio:",
    typical_center
)

Typical-county coverage ratio: 0.6705447637765135


In [3]:
adjacency_url = (
    "https://www2.census.gov/"
    "geo/docs/reference/county_adjacency/"
    "county_adjacency2024.txt"
)

adjacency = pd.read_csv(
    adjacency_url,
    sep="|",
    dtype={
        "County GEOID": str,
        "Neighbor GEOID": str
    }
)

print("Shape:", adjacency.shape)
display(adjacency.head())

Shape: (22196, 4)


,County Name,County GEOID,Neighbor Name,Neighbor GEOID
0,"Autauga County, AL",01001,"Autauga County, AL",01001
1,"Autauga County, AL",01001,"Chilton County, AL",01021
2,"Autauga County, AL",01001,"Dallas County, AL",01047
3,"Autauga County, AL",01001,"Elmore County, AL",01051
4,"Autauga County, AL",01001,"Lowndes County, AL",01085


In [4]:
mn_adjacency = adjacency[
    adjacency["County GEOID"].str.startswith("27")
].copy()

mn_adjacency = mn_adjacency[
    mn_adjacency["Neighbor GEOID"].str.startswith("27")
].copy()

mn_adjacency = mn_adjacency[
    mn_adjacency["County GEOID"]
    != mn_adjacency["Neighbor GEOID"]
].copy()

print(
    "Minnesota adjacency relationships:",
    len(mn_adjacency)
)

display(mn_adjacency.head())

Minnesota adjacency relationships: 444


,County Name,County GEOID,Neighbor Name,Neighbor GEOID
9066,"Aitkin County, MN",27001,"Carlton County, MN",27017
9067,"Aitkin County, MN",27001,"Cass County, MN",27021
9068,"Aitkin County, MN",27001,"Crow Wing County, MN",27035
9069,"Aitkin County, MN",27001,"Itasca County, MN",27061
9070,"Aitkin County, MN",27001,"Kanabec County, MN",27065


In [5]:
targets = [
    "Scott",
    "Pope",
    "Nicollet",
    "Norman",
    "Lac qui Parle"
]

target_fips = county[
    county["County"].isin(targets)
][
    [
        "County",
        "fips"
    ]
].copy()

target_neighbors = target_fips.merge(
    mn_adjacency,
    left_on="fips",
    right_on="County GEOID",
    how="left"
)

display(
    target_neighbors[
        [
            "County",
            "fips",
            "Neighbor Name",
            "Neighbor GEOID"
        ]
    ]
    .sort_values(
        ["County", "Neighbor Name"]
    )
)

,County,fips,Neighbor Name,Neighbor GEOID
0,Lac qui Parle,27073,"Big Stone County, MN",27011
1,Lac qui Parle,27073,"Chippewa County, MN",27023
2,Lac qui Parle,27073,"Swift County, MN",27151
3,Lac qui Parle,27073,"Yellow Medicine County, MN",27173
4,Nicollet,27103,"Blue Earth County, MN",27013
5,Nicollet,27103,"Brown County, MN",27015
6,Nicollet,27103,"Le Sueur County, MN",27079
7,Nicollet,27103,"Renville County, MN",27129
8,Nicollet,27103,"Sibley County, MN",27143
9,Norman,27107,"Becker County, MN",27005


In [6]:
hub_results = []

for target in targets:

    target_row = county[
        county["County"] == target
    ].iloc[0]

    fips_target = target_row["fips"]

    neighbor_fips = (
        mn_adjacency.loc[
            mn_adjacency["County GEOID"]
            == fips_target,
            "Neighbor GEOID"
        ]
        .dropna()
        .tolist()
    )

    pooled_fips = [
        fips_target
    ] + neighbor_fips

    pooled_data = county[
        county["fips"].isin(
            pooled_fips
        )
    ].copy()

    pooled_observed = (
        pooled_data["visits"].sum()
    )

    pooled_expected = (
        pooled_data["expected_visits"].sum()
    )

    pooled_ratio = (
        pooled_observed
        / pooled_expected
    )

    hub_results.append(
        {
            "County": target,
            "original_ratio":
                target_row["coverage_ratio"],
            "number_of_neighbors":
                len(neighbor_fips),
            "pooled_ratio":
                pooled_ratio,
            "change":
                pooled_ratio
                - target_row["coverage_ratio"]
        }
    )

hub_check = pd.DataFrame(
    hub_results
)

display(
    hub_check.sort_values(
        "original_ratio"
    )
)

,County,original_ratio,number_of_neighbors,pooled_ratio,change
0,Scott,0.111418,6,1.170120,1.058702
1,Pope,0.150499,6,0.436529,0.286030
2,Nicollet,0.165709,5,0.780458,0.614750
3,Norman,0.168078,4,0.592550,0.424472
4,Lac qui Parle,0.181299,4,0.375931,0.194633


In [7]:
hub_check["distance_from_typical_before"] = (
    abs(
        hub_check["original_ratio"]
        - typical_center
    )
)

hub_check["distance_from_typical_after"] = (
    abs(
        hub_check["pooled_ratio"]
        - typical_center
    )
)

hub_check["moves_toward_typical"] = (
    hub_check["distance_from_typical_after"]
    <
    hub_check["distance_from_typical_before"]
)

display(
    hub_check[
        [
            "County",
            "original_ratio",
            "pooled_ratio",
            "change",
            "moves_toward_typical"
        ]
    ]
)

,County,original_ratio,pooled_ratio,change,moves_toward_typical
0,Scott,0.111418,1.170120,1.058702,True
1,Pope,0.150499,0.436529,0.286030,True
2,Nicollet,0.165709,0.780458,0.614750,True
3,Norman,0.168078,0.592550,0.424472,True
4,Lac qui Parle,0.181299,0.375931,0.194633,True


Pooling each low-coverage county with its directly adjacent Minnesota counties moves all five ratios toward the typical-county value of approximately 0.67. Nicollet and Norman move close to typical, suggesting that cross-county utilization may explain much of their low county-only activity. Pope and Lac qui Parle improve but remain below typical, so their low coverage appears more persistent. Scott rises from 0.11 to 1.17, showing substantial surrounding regional activity, but this simple adjacency method likely overcorrects because large neighboring counties such as Hennepin and Dakota are treated as fully accessible. These results therefore support a full distance-based spatial-access analysis rather than final classification

In [8]:
hub_check["pooled_relative_to_typical"] = (
    hub_check["pooled_ratio"]
    / typical_center
)

hub_check["absolute_gap_after"] = (
    hub_check["pooled_ratio"]
    - typical_center
).abs()

display(
    hub_check[
        [
            "County",
            "original_ratio",
            "pooled_ratio",
            "pooled_relative_to_typical",
            "absolute_gap_after"
        ]
    ]
    .sort_values("absolute_gap_after")
)

,County,original_ratio,pooled_ratio,pooled_relative_to_typical,absolute_gap_after
3,Norman,0.168078,0.592550,0.883684,0.077995
2,Nicollet,0.165709,0.780458,1.163917,0.109914
1,Pope,0.150499,0.436529,0.651006,0.234016
4,Lac qui Parle,0.181299,0.375931,0.560636,0.294613
0,Scott,0.111418,1.170120,1.745030,0.499576
